In [0]:
# Load Bronze layer tables into Spark DataFrames

customers_df = spark.table("bronze.customers")
orders_df = spark.table("bronze.orders")
orders_item_df = spark.table("bronze.orders_item")
products_df = spark.table("bronze.products")
stores_df = spark.table("bronze.stores")


In [0]:
from pyspark.sql.functions import col

# Join orders with customers
orders_customers = orders_df.join(
    customers_df,
    "customer_id",
    "left"
)

# Rename duplicate column from customers
orders_customers = orders_customers.withColumnRenamed("city","customer_city")
orders_customers = orders_customers.withColumnRenamed("country","customer_country")
orders_customers = orders_customers.withColumnRenamed("state","customer_state")


# Join with stores table
orders_enriched = orders_customers.join(
    stores_df,
    "store_id",
    "left"
)

# Rename store city column
orders_enriched = orders_enriched.withColumnRenamed("city","store_city")
orders_enriched = orders_enriched.withColumnRenamed("country","store_country")
orders_enriched = orders_enriched.withColumnRenamed("state","store_state")



In [0]:
# Save enriched orders to Silver layer

orders_enriched.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver.orders_enriched")

In [0]:
# Join order_items with products

order_items_enriched = orders_item_df.join(
    products_df,
    "product_id",
    "left"
)

display(order_items_enriched)

In [0]:
order_items_enriched.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver.order_items_enriched")

In [0]:
orders_enriched_df = spark.table("silver.orders_enriched")
orders_enriched_df = orders_enriched_df.withColumnRenamed("brand","orders_brand")
orders_enriched_df = orders_enriched_df.withColumnRenamed("category","orders_category_new")
orders_enriched_df = orders_enriched_df.withColumnRenamed("cost","orders_cost")
orders_enriched_df = orders_enriched_df.withColumnRenamed("price","orders_price")
orders_enriched_df = orders_enriched_df.withColumnRenamed("product_name","orders_product_name")
orders_enriched_df = orders_enriched_df.withColumnRenamed("product_id","orders_product_id")
orders_enriched_df = orders_enriched_df.withColumnRenamed("state","orders_state")
orders_enriched_df = orders_enriched_df.withColumnRenamed("country","orders_country")
orders_enriched_df = orders_enriched_df.withColumnRenamed("city","orders_city")
orders_enriched_df = orders_enriched_df.withColumnRenamed("manager","store_manager")
orders_enriched_df = orders_enriched_df.withColumnRenamed("store_name","store_name")
orders_enriched_df = orders_enriched_df.withColumnRenamed("store_id","store_id")
orders_enriched_df = orders_enriched_df.withColumnRenamed("store_state","store_state")
orders_enriched_df = orders_enriched_df.withColumnRenamed("store_country","store_country")
orders_enriched_df = orders_enriched_df.withColumnRenamed("store_city","store_city")
orders_enriched_df = orders_enriched_df.withColumnRenamed("customer_segment","orders_category")
items_enriched_df = spark.table("silver.order_items_enriched")
items_enriched_df = items_enriched_df.withColumnRenamed("brand","product_brand")
items_enriched_df = items_enriched_df.withColumnRenamed("category","product_category")



sales_fact = orders_enriched_df.join(
    items_enriched_df,
    "order_id",
    "inner"
)

display(sales_fact)

In [0]:
from pyspark.sql.functions import col

sales_fact = sales_fact.withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

In [0]:
sales_fact.write.format("delta") \
.mode("overwrite") \
.option("mergeSchema","true")\
.saveAsTable("silver.sales_fact")